# Discover Resonances and Estimate Q Factors

In mechanical suspensions, mirror test masses, and seismic isolation systems, high-Q resonant modes amplify narrow-band noise and dictate control loop stability. Accurately extracting natural resonance frequencies $f_n$ and quality factors $Q$ from drive-response measurements is essential for modal modeling.

**What you will achieve:**
1. Generate synthetic impulse drive and multi-resonance mechanical response signals (73 Hz, 211 Hz, 389 Hz).
2. Estimate the input-output complex transfer function across transient excitation windows.
3. Automatically detect resonant peak candidates using `find_peaks()`.
4. Fit local complex resonant models in the frequency domain to estimate $f_n$ and $Q$.
5. Perform ringdown amplitude envelope fitting in the time domain to extract damping decay constants $\tau_A$ and reverberation time $\text{RT}_{60}$.
6. Cross-validate frequency-domain and time-domain Q estimates.

**Data type**: Synthetic impulse drive and 3-mode mechanical response (30 s, 2048 Hz).

## Environment Setup

In [ ]:
import json
import os
import tempfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from astropy import units as u
from scipy import signal
from scipy.optimize import curve_fit

import gwexpy
from gwexpy.frequencyseries import FrequencySeries
from gwexpy.timeseries import TimeSeries

output_dir = Path(os.environ.get("GWEXPY_DOCS_OUTPUT_DIR") or tempfile.mkdtemp(prefix="gwexpy-t3-"))
output_dir.mkdir(parents=True, exist_ok=True)
(output_dir / "figures").mkdir(exist_ok=True)
(output_dir / "tables").mkdir(exist_ok=True)
print(f"Output directory: {output_dir}")

## Synthetic Impulse Excitation and Resonant Fixture

- Duration: 30 s, $f_s = 2048\,\text{Hz}$ ($N = 61440$ samples).
- Impulse excitation applied at $t = 2.0\,\text{s}$.
- 3 isolated resonant modes:
  - Mode 1: $f_n = 73.0\,\text{Hz}, Q = 40.0, A = 0.6$
  - Mode 2: $f_n = 211.0\,\text{Hz}, Q = 80.0, A = 0.4$
  - Mode 3: $f_n = 389.0\,\text{Hz}, Q = 120.0, A = 0.25$

In [ ]:
fs = 2048.0
dt = 1.0 / fs
duration = 30.0
n_samples = int(fs * duration)
t = np.arange(n_samples) * dt

# Drive: impulse at t = 2.0 s (sample 4096)
drive = np.zeros(n_samples)
drive_idx = int(2.0 * fs)
drive[drive_idx] = 1.0

# Synthesize response from exact discrete oscillator equations
response = np.zeros(n_samples)
truth_modes = [
    {"mode_id": "M1", "fn": 73.0, "Q": 40.0, "A": 0.6},
    {"mode_id": "M2", "fn": 211.0, "Q": 80.0, "A": 0.4},
    {"mode_id": "M3", "fn": 389.0, "Q": 120.0, "A": 0.25},
]

for m in truth_modes:
    fn = m["fn"]
    Q = m["Q"]
    A = m["A"]
    gamma = np.pi * fn / Q
    fd = np.sqrt(fn**2 - (gamma / (2 * np.pi))**2)
    # Discrete transfer function poles: r * exp(+- j*theta)
    r = np.exp(-gamma / fs)
    theta = 2 * np.pi * fd / fs
    b = [0, A * r * np.sin(theta)]
    a = [1, -2 * r * np.cos(theta), r**2]
    mode_resp = signal.lfilter(b, a, drive)
    response += mode_resp

# Add small realistic sensor noise (sigma = 1e-5 V)
rng = np.random.default_rng(2026091603)
response += rng.normal(0, 1e-5, n_samples)

ts_drive = TimeSeries(drive, dt=dt*u.s, unit=u.V, name="DRIVE")
ts_resp = TimeSeries(response, dt=dt*u.s, unit=u.V, name="RESPONSE")
print("Constructed drive and response TimeSeries")

## Transfer Function Estimation and Peak Discovery

We take an 8-second post-trigger window ($t = [2.0, 10.0)\,\text{s}$) to compute the complex frequency response $H(f) = Y(f) / X(f)$ and detect resonant candidate peaks.

In [ ]:
# Post-trigger analysis window
w_start = 2.0 * u.s
w_end = 10.0 * u.s
sub_drive = ts_drive.crop(w_start, w_end)
sub_resp = ts_resp.crop(w_start, w_end)

# Complex FFT
fft_drive = np.fft.rfft(sub_drive.value)
fft_resp = np.fft.rfft(sub_resp.value)
freqs = np.fft.rfftfreq(len(sub_drive.value), d=dt)

# Transfer function H(f)
H = fft_resp / np.where(np.abs(fft_drive) > 1e-6, fft_drive, 1e-6)
H_mag = np.abs(H)

fs_mag = FrequencySeries(H_mag, df=(freqs[1]-freqs[0])*u.Hz, unit=u.dimensionless_unscaled, name="TRANSFER_MAG")
peaks, props = fs_mag.find_peaks(height=5.0, distance=30 * u.Hz)

detected_candidates = []
for f_val in peaks.frequencies.value:
    if 50.0 <= f_val <= 450.0:
        detected_candidates.append(float(f_val))

print(f"Detected resonance candidates: {detected_candidates}")

## Frequency-Domain Resonant Peak Fitting

Around each candidate peak, we fit the local complex resonant amplitude:
$$|H(f)|^2 = \frac{A^2}{(1 - (f/f_n)^2)^2 + (f / (Q f_n))^2} + B$$

In [ ]:
def resonance_model(f, fn, Q, A, B):
    denom = (1.0 - (f / fn)**2)**2 + (f / (Q * fn))**2
    return (A**2) / denom + B

modes_records = []

for idx, f_cand in enumerate(detected_candidates):
    # Fit window +- 6 Hz around candidate
    fit_mask = (freqs >= f_cand - 6.0) & (freqs <= f_cand + 6.0)
    f_fit = freqs[fit_mask]
    power_fit = (H_mag[fit_mask])**2

    p0 = [f_cand, 50.0, np.max(H_mag[fit_mask]) / 50.0, 0.0]
    bounds = ([f_cand - 5.0, 5.0, 0.0, 0.0], [f_cand + 5.0, 500.0, 100.0, 10.0])

    popt, _ = curve_fit(resonance_model, f_fit, power_fit, p0=p0, bounds=bounds)
    fn_est, q_est, A_est, B_est = popt

    # Time-domain ringdown: bandpass around fn +- 4 Hz and fit envelope
    sos_bp = signal.butter(2, [fn_est - 4.0, fn_est + 4.0], btype="bandpass", fs=fs, output="sos")
    resp_bp = signal.sosfilt(sos_bp, response[drive_idx:])
    t_rd = t[:len(resp_bp)]

    # Hilbert analytic envelope
    env = np.abs(signal.hilbert(resp_bp))
    # Fit interval: t = 0.2 s to 1.5 s
    rd_mask = (t_rd >= 0.2) & (t_rd <= 1.5)
    t_fit_rd = t_rd[rd_mask]
    log_env = np.log(np.maximum(env[rd_mask], 1e-12))

    slope, intercept = np.polyfit(t_fit_rd, log_env, 1)
    tau_A = -1.0 / slope
    t60_s = np.log(1000.0) * tau_A
    q_time = np.pi * fn_est * tau_A

    modes_records.append({
        "mode_id": f"MODE_{idx+1:02d}",
        "candidate_hz": float(f_cand),
        "fn_fit_hz": float(fn_est),
        "fd_fit_hz": float(fn_est * np.sqrt(max(1.0 - 1.0/(4*q_est**2), 0.0))),
        "q_freq": float(q_est),
        "tau_amp_s": float(tau_A),
        "q_time": float(q_time),
        "t60_s": float(t60_s),
        "fit_start_hz": float(f_cand - 6.0),
        "fit_end_hz": float(f_cand + 6.0),
        "status": "valid"
    })

modes_df = pd.DataFrame(modes_records)
modes_df.to_csv(output_dir / "tables/modes.csv", index=False)
print("Fitted mode parameters saved to tables/modes.csv:")
print(modes_df[["mode_id", "fn_fit_hz", "q_freq", "q_time", "t60_s"]])

## Diagnostic Plots and Verification

In [ ]:
# Plot 1: Transfer function and detected candidates
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.semilogy(freqs, H_mag, label="|H(f)|", color="tab:blue")
for cand in detected_candidates:
    ax.axvline(cand, color="crimson", ls="--", alpha=0.7, label=f"Mode @ {cand:.1f} Hz")
ax.set_xlim(20, 500)
ax.set_xlabel("Frequency [Hz]")
ax.set_ylabel("Transfer Magnitude |H(f)|")
ax.set_title("Impulse Transfer Function with Detected Resonances")
ax.grid(True, alpha=0.3)
plt.tight_layout()
fig.savefig(output_dir / "figures/frequency_domain_fit.png", dpi=150)
plt.close(fig)

# Quality Checks
cand_ok = bool(len(detected_candidates) == 3)

# Accuracy checks against truth
fn_errors = []
q_errors = []
cross_errors = []

for idx, tr in enumerate(truth_modes):
    row = modes_df.iloc[idx]
    fn_err = abs(row["fn_fit_hz"] - tr["fn"]) / tr["fn"]
    q_err = abs(row["q_freq"] - tr["Q"]) / tr["Q"]
    cross_err = abs(row["q_freq"] - row["q_time"]) / row["q_freq"]
    fn_errors.append(fn_err)
    q_errors.append(q_err)
    cross_errors.append(cross_err)

fn_ok = bool(max(fn_errors) < 0.01)      # < 1% error on frequency
q_ok = bool(max(q_errors) < 0.15)        # < 15% error on Q
cross_ok = bool(max(cross_errors) < 0.20) # < 20% consistency between freq and time domains

metrics = {
    "status": "passed" if (cand_ok and fn_ok and q_ok and cross_ok) else "failed",
    "checks": {
        "resonance_candidates": {"passed": cand_ok, "count": len(detected_candidates)},
        "resonance_fn_recovery": {"passed": fn_ok, "max_fn_error": float(max(fn_errors))},
        "resonance_q_recovery": {"passed": q_ok, "max_q_error": float(max(q_errors))},
        "resonance_crosscheck": {"passed": cross_ok, "max_cross_error": float(max(cross_errors))},
        "resonance_rt60_definition": {"passed": True}
    }
}

with open(output_dir / "validation-metrics.json", "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2)

settings = {
    "tutorial_id": "T3",
    "fs_hz": 2048.0,
    "truth_modes": truth_modes
}
with open(output_dir / "analysis-settings.json", "w", encoding="utf-8") as f:
    json.dump(settings, f, indent=2)

print("Validation metrics:")
print(json.dumps(metrics, indent=2))
assert metrics["status"] == "passed", "T3 verification failed!"